# NOM / IR / PE — 3×4 Experiment
3 agents, 4 items

In [1]:
# ── Colab セットアップ（最初に1回だけ実行）──────────────────────────
!git clone https://github.com/shiro-seminar/NOM-matching.git

import sys
sys.path.insert(0, "/content/NOM-matching")

Cloning into 'NOM-matching'...
remote: Enumerating objects: 92, done.
remote: Counting objects: 100% (92/92), done.
remote: Compressing objects: 100% (69/69), done.
remote: Total 92 (delta 37), reused 74 (delta 19), pack-reused 0 (from 0)
Receiving objects: 100% (92/92), 78.98 KiB | 4.65 MiB/s, done.
Resolving deltas: 100% (37/37), done.


In [5]:
# コードを更新したいときはここを実行
!git -C /content/NOM-matching pull

remote: Enumerating objects: 5, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (1/1), done.
remote: Total 3 (delta 2), reused 3 (delta 2), pack-reused 0 (from 0)
Unpacking objects: 100% (3/3), 315 bytes | 315.00 KiB/s, done.
From https://github.com/shiro-seminar/NOM-matching
   8f41aa1..76796f6  main       -> origin/main
Updating 8f41aa1..76796f6
Fast-forward
 nom_ir_pe_3x4_experiment.ipynb | 10 ++--------
 1 file changed, 2 insertions(+), 8 deletions(-)


In [2]:
import torch
from nom_ir_pe_3x4.config      import Config
from nom_ir_pe_3x4.train       import train
from nom_ir_pe_3x4.evaluate    import evaluate_mechanism, print_table
from nom_ir_pe_3x4.benchmarks  import BENCHMARKS
from nom_ir_pe_3x4.data_gen    import sample_batch
from nom_ir_pe_3x4.model       import AllocationNet
from nom_ir_pe_3x4.allocations import ir_pe_mask

## 1. Config

In [3]:
cfg = Config(
    steps      = 50_000,   # 短くしたいなら 5_000 など
    batch_size = 64,
    device     = "cuda" if torch.cuda.is_available() else "cpu",
    seed       = 42,
)
print(cfg)
print(f"device: {cfg.device}")

Config(num_agents=3, num_items=4, v_min=0.0, v_max=1.0, hidden=256, depth=4, batch_size=64, steps=50000, lr=0.0003, grad_clip=1.0, seed=42, S=8, M=8, temperature=1.0, lambda_nom=0.0, rho=1.0, rho_mult=1.005, rho_max=200.0, dual_update_every=100, nom_target=0.005, welfare_weight=0.02, device='cuda')
device: cuda


## 2. Train

In [4]:
net = train(cfg)

step=     1  welfare=2.5726  nom=0.00000  λ=0.000  ρ=1.00  elapsed=1s
step=   200  welfare=2.6565  nom=0.00000  λ=0.000  ρ=1.00  elapsed=12s
step=   400  welfare=2.6200  nom=0.00000  λ=0.000  ρ=1.00  elapsed=23s
step=   600  welfare=2.5140  nom=0.00000  λ=0.000  ρ=1.00  elapsed=34s
step=   800  welfare=2.5396  nom=0.00000  λ=0.000  ρ=1.00  elapsed=45s
step=  1000  welfare=2.5581  nom=0.00000  λ=0.000  ρ=1.00  elapsed=56s
step=  1200  welfare=2.6318  nom=0.00000  λ=0.000  ρ=1.00  elapsed=67s
step=  1400  welfare=2.5399  nom=0.00020  λ=0.000  ρ=1.00  elapsed=78s
step=  1600  welfare=2.5977  nom=0.00000  λ=0.000  ρ=1.00  elapsed=89s
step=  1800  welfare=2.6639  nom=0.00007  λ=0.000  ρ=1.00  elapsed=100s
step=  2000  welfare=2.6664  nom=0.00000  λ=0.000  ρ=1.00  elapsed=111s
step=  2200  welfare=2.5579  nom=0.00000  λ=0.000  ρ=1.00  elapsed=122s
step=  2400  welfare=2.5376  nom=0.00000  λ=0.000  ρ=1.00  elapsed=132s
step=  2600  welfare=2.5716  nom=0.00000  λ=0.000  ρ=1.00  elapsed=143s
st

## 3. Evaluate — benchmarks vs learned net

In [5]:
eval_cfg = Config(batch_size=500, device=cfg.device)
torch.manual_seed(0)

batch = sample_batch(eval_cfg)
v, endow_idx, U = batch["v"], batch["endow_idx"], batch["U"]
wmax_w = U.sum(1).max(1).values

results = []
for bname, bfn in BENCHMARKS.items():
    print(f"Evaluating {bname}...")
    results.append(evaluate_mechanism(bname, bfn, eval_cfg, v, endow_idx, U, wmax_w))

net.eval()
def net_mech(cfg_, v_, ei_, U_):
    mask = ir_pe_mask(cfg_, U_, ei_)
    return net(v_, mask=mask, temperature=1e-3)

results.append(evaluate_mechanism("LearnedNet", net_mech, eval_cfg, v, endow_idx, U, wmax_w, is_nn=True))

print_table(results)

Evaluating Endowment...
Evaluating WMAX-IR...
Evaluating WMAX-PE...
Evaluating WMAX-IR-PE...
Evaluating TTC-general...
Evaluating Random-IR-PE...

─────────────────────────────────────────────────────────────────────────────────
Mechanism         Welfare  W/WMAX  IR-viol%     PE%   IR∩PE%  NOM-mean  NOM-viol%
─────────────────────────────────────────────────────────────────────────────────
Endowment          1.9712   0.661       0.0    27.8     31.8   0.00000        0.0
WMAX-IR            2.6686   0.895       0.0   100.0     79.2   0.00020        0.6
WMAX-PE            2.9826   1.000      84.8   100.0      8.6   0.17528       95.8
WMAX-IR-PE         2.5904   0.868       0.0    96.0    100.0   0.00000        0.0
TTC-general        2.1647   0.726       0.0    42.0     39.2   0.00000        0.0
Random-IR-PE       2.5368   0.851       0.0    96.0    100.0   0.00000        0.2
LearnedNet         2.5887   0.868       0.0    96.0    100.0   0.00000        0.0
─────────────────────────────────

## 4. (Optional) チェックポイントを保存 / 読み込み

In [7]:
# Google Drive にモデルを保存したい場合
from google.colab import drive
drive.mount("/content/drive")

import shutil
shutil.copy("alloc_net_3x4.pt", "/content/drive/MyDrive/alloc_net_3x4.pt")
print("saved to Drive")

Mounted at /content/drive
saved to Drive


In [ ]:
# Drive から読み込む場合
CKPT = "/content/drive/MyDrive/alloc_net_3x4.pt"
ckpt = torch.load(CKPT, map_location=cfg.device)
net2 = AllocationNet(Config(**ckpt["cfg"]))
net2.load_state_dict(ckpt["state_dict"])
net2.eval()
print(f"loaded: step={ckpt.get('step', 'final')}")